# 01 · Design checks: SRM, balance, power

Before any effect is estimated: did randomisation deliver the cells we asked for, are the arms
comparable, and could this experiment have detected the effect sizes we care about?

Read with: [`docs/learn/01_potential_outcomes.md`](../docs/learn/01_potential_outcomes.md) §1.5 and [`docs/learn/02_cuped_and_regression_adjustment.md`](../docs/learn/02_cuped_and_regression_adjustment.md).

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import Image, display
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from tierlift.config import REPO_ROOT, load_config
from tierlift.simulate import SimulatedData
cfg = load_config()
DATA, RESULTS, FIGS = REPO_ROOT / "data", REPO_ROOT / "results", REPO_ROOT / "reports" / "figures"
assert DATA.exists(), "run `make all` (or `tierlift run-all`) first: the notebooks read data/ and results/"
data = SimulatedData.load(DATA)
e, truth = data.experiment, data.truth.set_index("client_id")
primary, control = cfg.analysis.primary_contrast
def fig(name): display(Image(filename=str(FIGS / name), width=900))
print(f"seed {cfg.seed}, {len(e):,} clients, cells {cfg.experiment.cells}")

seed 13, 5,000 clients, cells {'holdout': 0.3, 'nudge': 0.3, 'incentive': 0.4}


## 1. Sample-ratio mismatch

In [2]:
from tierlift.experiment.design import srm_test, balance_table, power_table, mde
srm = srm_test(e["cell"], cfg.experiment.cells); srm

{'observed': {'holdout': 1503, 'nudge': 1498, 'incentive': 1999},
 'expected': {'holdout': 1500.0, 'nudge': 1500.0, 'incentive': 2000.0},
 'chi2': 0.009166666666666667,
 'p_value': 0.9954271541102877}

## 2. Covariate balance (standardised mean differences)

In [3]:
from tierlift.validation.checks import BALANCE_COVARIATES
bal = balance_table(e, BALANCE_COVARIATES); display(bal)
print("max |SMD|:", bal.filter(like="smd").abs().max().max().round(3), "(rule of thumb: < 0.1 is balanced)")

,cell,covariate,mean_treated,mean_holdout,smd
0,incentive,pre_revenue,"58,816.4738","58,763.0705",0.0011
1,incentive,pre_volume,"65,672,939.6597","65,434,803.3867",0.0035
2,incentive,api_share,0.4217,0.4273,-0.0177
3,incentive,engagement_z,0.0334,-0.0284,0.0624
4,incentive,aum_flow_z,-0.0215,0.0104,-0.0320
5,incentive,tenure_months,35.6688,36.9195,-0.0483
6,incentive,has_account_manager,0.2101,0.1969,0.0327
7,incentive,maker_ratio,0.4207,0.4215,-0.0048
8,nudge,pre_revenue,"59,230.1357","58,763.0705",0.0091
9,nudge,pre_volume,"66,240,682.6557","65,434,803.3867",0.0111


max |SMD|: 0.062 (rule of thumb: < 0.1 is balanced)


## 3. Minimum detectable effect, with and without CUPED

ρ is the pre/post revenue correlation in the holdout; CUPED multiplies the variance by (1 − ρ²).

In [4]:
h = e[e["cell"] == control]
rho = float(np.corrcoef(h["pre_revenue"], h["exp_revenue"])[0, 1])
pw = power_table(float(h["exp_revenue"].std(ddof=1)), float(h["exp_revenue"].mean()), rho, len(e), cfg.experiment.cells, cfg.analysis.alpha)
display(pw)
print(f"rho = {rho:.3f}; CUPED is worth a sample {1 / (1 - rho**2):.2f}x as large")

,cell,n_treated,n_holdout,mde_raw_pct,mde_cuped_pct,rho,equivalent_sample_multiplier
0,nudge,1500,1500,0.1241,0.0842,0.7349,2.1746
1,incentive,2000,1500,0.1161,0.0787,0.7349,2.1746


rho = 0.735; CUPED is worth a sample 2.17x as large


## Check your understanding

1. The MDE on raw revenue is ≈ 12%. The planted blanket effect is −3%. What does that tell you about a *population-level* significance test, and why does the project lean on the targeted band and on AIPW instead?
2. Suppose the holdout had been 10% instead of 30%. Which number in the power table moves most, and in which direction?